# QUELLE RÉPARTITION DES PRÊTS À LA MÉDIATHÈQUE ?

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import kiblib.utils.lucas_sns_params
from kiblib.utils.db import DbConn

In [2]:
maj = dt.today().strftime('%d-%m-%Y à %Hh%M' )
print(f"Dernière mise à jour le {maj}")

Dernière mise à jour le 06-11-2026 à 10h54


In [3]:
db_conn = DbConn().create_engine()

In [4]:
query = """SELECT *
FROM koha_prod.items i
LEFT JOIN koha_prod.authorised_values av ON av.authorised_value = i.location"""

In [5]:
items = pd.read_sql(query,db_conn)

In [6]:
cols2print = ["notforloan","damaged","damaged_on","barcode","location","ccode"]

In [7]:
# EXEMPLAIRES
items["Exemplaires"] = 1

In [8]:
# ACQUISITIONS
items.loc[items["dateaccessioned"].astype("Datetime64").between("2026-01-01","2026-12-31"),"Acquisitions"] = 1

In [9]:
# DOCUMENTS A DISPOSITION DU PUBLIC
pret_direct = ["MED0C","MED1A","MED2A","MED3A"]
pret_indirect = ["MED0A","MED2C","MED2D","MED3C","MED3D","MED3E","MED3F","MED3G","MED3H","MED3I"]
consultation = ["MED0B","MED2B"]

# Empruntables
items.loc[items["notforloan"]==0,"Exemplaires empruntables"] = 1

# En accès direct
items.loc[~(items["location"].isin(pret_indirect))&
          ~(items["location"].isin(consultation)),
          "Exemplaires en accès direct"] = 1

# Consultation uniquement
items.loc[items["location"].isin(consultation),
         "Exemplaires en consultation sur place uniquement"] = 1

# En accès indirect
items.loc[items["location"].isin(pret_indirect),
         "Exemplaires en accès indirect"] = 1

In [10]:
# DOCUMENTS DANS LE CIRCUIT INTERNE

items.loc[items["notforloan"]==-1,"Exemplaires en commande"] = 1
items.loc[items["notforloan"]==-2,"Exemplaires en traitement"] = 1
items.loc[items["damaged"]!=0,"Exemplaires abîmés"] = 1
items.loc[items["notforloan"]==-4,"Exemplaires en réparation"] = 1
items.loc[items["notforloan"]==5,"Exemplaires en reliure"] = 1
items.loc[items["itemlost"]==2,"Exemplaires perdus"] = 1
items.loc[items["itemlost"]==1,"Exemplaires non-restitués"] = 1

In [11]:
# DOCUMENTS ÉLIMINÉS

items.loc[(items["notforloan"]==4) &
          (items["itemlost"]==1),"Exemplaires éliminés non-restitués"] = 1

items.loc[(items["notforloan"]==4)&
          (items["itemlost"]==2),
          "Exemplaires éliminés perdus"] = 1


items.loc[(items["notforloan"]==4)&
          (items["withdrawn"]==1),
          "Exemplaires éliminés desherbés"] = 1


items.loc[(items["notforloan"]==4)&
          (items["damaged"]==1),
          "Exemplaires éliminés abîmés"] = 1

In [12]:
site_med = ["MED0B",
            "MED0C",
            "MED1A",
            "MED2A",
            "MED2B",
            "MED2C",
            "MED2D",
            "MED3A",
            "MED3B",
            "MED3C",
            "MED3D",
            "MED3E",
            "MED3F",
            "MED3G",
            "MED3H",
            "MED3I",
            "MED3J",
            "MED3K",
            "MED3L",
            "MED3M",
            "MED3N",
            "MED3O",
            "MED3P",
            "MED3Q",
            "MED3R",
            "MED3S",
            "MED3T"]

site_bus = ["BUS1A"]

site_col = ["MED0A"]

In [13]:
# SITES

# Médiathèque
items.loc[(items["homebranch"]=="MED")&
          (items["location"].isin(site_med)),"Site"] = "Médiatèque"

# Zébron
items.loc[items["homebranch"]=="BUS","Site"] = "Zébron"

# Collectivités
items.loc[items["location"]=="MED0A","Site"] = "Collectivités"

In [14]:
cols2keep = ["Site",
             "ccode",
             "location",
             "lib",
             'Exemplaires', 'Acquisitions',
             'Exemplaires empruntables', 'Exemplaires en accès direct',
             'Exemplaires en consultation sur place uniquement',
             'Exemplaires en accès indirect', 'Exemplaires en commande',
             'Exemplaires en traitement', 'Exemplaires abîmés',
             'Exemplaires en réparation', 'Exemplaires en reliure',
             'Exemplaires perdus', 'Exemplaires non-restitués',
             'Exemplaires éliminés non-restitués', 'Exemplaires éliminés perdus',
             'Exemplaires éliminés desherbés', 'Exemplaires éliminés abîmés']

indicateurs = ['Exemplaires', 'Acquisitions',
              'Exemplaires empruntables', 'Exemplaires en accès direct',
              'Exemplaires en consultation sur place uniquement',
              'Exemplaires en accès indirect', 'Exemplaires en commande',
              'Exemplaires en traitement', 'Exemplaires abîmés',
              'Exemplaires en réparation', 'Exemplaires en reliure',
              'Exemplaires perdus', 'Exemplaires non-restitués',
              'Exemplaires éliminés non-restitués', 'Exemplaires éliminés perdus',
              'Exemplaires éliminés desherbés', 'Exemplaires éliminés abîmés']

In [15]:
analyse = items[cols2keep]

In [16]:
analyse

,Site,ccode,location,lib,Exemplaires,Acquisitions,Exemplaires empruntables,Exemplaires en accès direct,Exemplaires en consultation sur place uniquement,Exemplaires en accès indirect,...,Exemplaires en traitement,Exemplaires abîmés,Exemplaires en réparation,Exemplaires en reliure,Exemplaires perdus,Exemplaires non-restitués,Exemplaires éliminés non-restitués,Exemplaires éliminés perdus,Exemplaires éliminés desherbés,Exemplaires éliminés abîmés
0,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
2,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Collectivités,JABEFAE,MED0A,Magasin collectivités,1,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268082,Médiatèque,PPEPRZZ,MED3T,Mag Patrimoine,1,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
268083,Médiatèque,PPEPRZZ,MED3T,Mag Patrimoine,1,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
268084,Médiatèque,PPEPRZZ,MED3T,Mag Patrimoine,1,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
268085,Médiatèque,PPEPRZZ,MED3T,Mag Patrimoine,1,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# FILTRE SUR LES EXEMPLAIRES EN PRÊT DIRECT 
# SITE MEDIATHÈQUE
analyse_med = analyse[(analyse["Site"]=="Médiatèque")
                     #&(analyse["location"].isin(pret_direct))
                     ]

In [36]:
analyse_med = analyse_med.pivot_table(index=["location","lib"],values=indicateurs,aggfunc=sum)

In [37]:
analyse_med.reindex(cols2keep,axis=1)[["Exemplaires abîmés","Exemplaires en réparation","Exemplaires éliminés abîmés"]]

,,Exemplaires abîmés,Exemplaires en réparation,Exemplaires éliminés abîmés
location,lib,,,
MED0B,Espace Périodiques,0.0,0.0,0.0
MED0C,Rez-de-chaussée,21.0,4.0,14.0
MED1A,1er étage,589.0,71.0,490.0
MED2A,2e étage - Jeunesse,1467.0,239.0,1144.0
MED2B,2e étage - Salle d'étude,0.0,0.0,0.0
MED2C,Magasin jeunesse,4.0,2.0,1.0
MED2D,Magasin disco 2e étage,0.0,0.0,0.0
MED3A,3e étage,39.0,20.0,15.0
MED3B,Espace Multimédia,0.0,0.0,0.0


- Exemplaires
    * Total
    * Empruntables
        * En accès direct
        * En accès indirect
    * Consultation sur place uniquement

- Acquisitions
- Exemplaires dans le circuit du document
    * En commande
    * En traitement
    * En réparation
    * Perdus
    * Non restitués

- Exemplaires sortis des collections
    * Désherbé
    * Non-restitué
    * Perdus
    * Abîmés

# DE QUOI SE COMPOSENT LES COLLECTIONS ?

## Quel est le nombre d'exemplaires par espaces ?

In [70]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires",ascending=False)

,,Exemplaires
location,lib,
MED1A,1er étage,67423
MED2A,2e étage - Jeunesse,40388
MED3E,Mag Pôle adultes °8,21011
MED2D,Magasin disco 2e étage,15847
MED3A,3e étage,15366
MED3I,Mag Patrimoine °8,14317
MED0C,Rez-de-chaussée,10959
MED3C,Magasin disco 3e étage,10713
MED3T,Mag Patrimoine,7165


In [84]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires empruntables",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires empruntables",ascending=False).loc[analyse_med["Exemplaires empruntables"]!=0]

,,Exemplaires empruntables
location,lib,
MED1A,1er étage,63580
MED2A,2e étage - Jeunesse,34311
MED3E,Mag Pôle adultes °8,20969
MED3A,3e étage,14259
MED2D,Magasin disco 2e étage,11654
MED0C,Rez-de-chaussée,10323
MED3C,Magasin disco 3e étage,3064
MED3D,Mag Bibliothèque professionnelle,2071
MED3H,Mag Pôle adultes BD,1770


In [81]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en accès direct",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en accès direct",ascending=False).loc[analyse_med["Exemplaires en accès direct"]!=0]

,,Exemplaires en accès direct
location,lib,
MED1A,1er étage,67423
MED2A,2e étage - Jeunesse,40388
MED3A,3e étage,15366
MED0C,Rez-de-chaussée,10959
MED3T,Mag Patrimoine,7165
MED3J,Mag Patrimoine °4,6239
MED3N,Mag Patrimoine BR 4,3701
MED3M,Mag Patrimoine BR 8,3575
MED3O,Mag Patrimoine RES 8,1998


In [82]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en accès indirect",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en accès indirect",ascending=False).loc[analyse_med["Exemplaires en accès indirect"]!=0]

,,Exemplaires en accès indirect
location,lib,
MED3E,Mag Pôle adultes °8,21011
MED2D,Magasin disco 2e étage,15847
MED3I,Mag Patrimoine °8,14317
MED3C,Magasin disco 3e étage,10713
MED3D,Mag Bibliothèque professionnelle,2123
MED3H,Mag Pôle adultes BD,1798
MED3G,Mag Pôle adultes fol,1010
MED2C,Magasin jeunesse,963
MED3F,Mag Pôle adultes °4,554


In [83]:
analyse_med.pivot_table(index=["location","lib"],
                        values="Exemplaires en consultation sur place uniquement",
                        fill_value=np.nan,
                        aggfunc=sum).dropna().sort_values(by="Exemplaires en consultation sur place uniquement",ascending=False).loc[analyse_med["Exemplaires en consultation sur place uniquement"]!=0]

,,Exemplaires en consultation sur place uniquement
location,lib,
MED2B,2e étage - Salle d'étude,1478
MED0B,Espace Périodiques,5
